# Hyperparameter Tuning (W&B)

Systematically search over learning rate, regularization, and model capacity using **Weights & Biases Bayesian sweeps** on the TinyShakespeare character-level GPT.

## Prerequisites

- Run `2. Preprocessing.ipynb` first — outputs in `data/artifacts/`.
- Optional: review `3. Baseline Model Architecture.ipynb` for the same GPT design.
- Install W&B: `pip install wandb`, then `wandb login`.

## Workflow

| Step | Section | What happens |
|------|---------|--------------|
| 1 | Setup | Imports, device, default hyperparameters (`BASE_CONFIG`) |
| 2 | Load data | Tokenized train/val splits + batch sampler |
| 3 | Model | Decoder-only GPT (same architecture as baseline) |
| 4 | Training | Optimizer, LR schedule, W&B `train()` callback |
| 5 | Sweep config | Define Bayesian search space |
| 6 | Launch | Create sweep → run `wandb.agent()` |
| 7 | Results | Compare runs on the W&B dashboard |

Each trial logs metrics, tracks best `val/loss`, and saves a checkpoint + artifact.


## Setup

Import dependencies, resolve the project root, and define `BASE_CONFIG`.

These are the **fallback defaults** for any hyperparameter the sweep does not override. Inside `train()`, sweep values from `wandb.config` are merged on top.


In [1]:
from pathlib import Path
import random
import time
import math
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import wandb
except ImportError as e:
    raise ImportError("Install wandb: pip install wandb") from e

cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"Could not locate a 'data' directory from cwd: {cwd}")

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = DATA_DIR / "artifacts"

BASE_CONFIG = {
    "seed": 42,
    "block_size": 128,
    "batch_size": 32,
    "max_steps": 2000,
    "eval_interval": 200,
    "learning_rate": 3e-4,
    "weight_decay": 0.1,
    "betas": (0.9, 0.95),
    "warmup_steps": 100,
    "min_lr": 1e-5,
    "n_layers": 4,
    "d_model": 128,
    "n_heads": 4,
    "d_ff": 4,
    "dropout": 0.2,
    "wandb_project": "genre-story-generator",
    "wandb_entity": None,
    "wandb_group": "sweep_lr_dropout",
    "wandb_job_type": "training-sweep",
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
set_seed(BASE_CONFIG["seed"])


Using device: cuda


## Load Artifacts

Load preprocessed token IDs from notebook 2. The `get_batch` helper returns:

- **x** — input tokens `(batch_size, block_size)`
- **y** — next-token targets, shifted by one position


In [2]:
train_ids_path = ARTIFACTS_DIR / "train_ids.pt"
val_ids_path = ARTIFACTS_DIR / "val_ids.pt"
vocab_path = ARTIFACTS_DIR / "char_vocab.json"

assert train_ids_path.exists(), f"Missing {train_ids_path}. Run preprocessing notebook first."
assert val_ids_path.exists(), f"Missing {val_ids_path}. Run preprocessing notebook first."
assert vocab_path.exists(), f"Missing {vocab_path}. Run preprocessing notebook first."

train_ids = torch.load(train_ids_path)
val_ids = torch.load(val_ids_path)

with open(vocab_path, "r", encoding="utf-8") as f:
    vocab_payload = json.load(f)

vocab_size = vocab_payload["vocab_size"]
print("train_ids:", train_ids.shape, "| val_ids:", val_ids.shape, "| vocab:", vocab_size)


def get_batch(split: str, batch_size: int, block_size: int, device: str = device):
    if split == "train":
        data = train_ids
    elif split == "val":
        data = val_ids
    else:
        raise ValueError("split must be 'train' or 'val'")

    if len(data) <= block_size:
        raise ValueError(f"{split} split too short for block_size={block_size}")

    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


train_ids: torch.Size([1003854]) | val_ids: torch.Size([55770]) | vocab: 65


## Model Architecture

Compact **decoder-only GPT** (identical design to notebook 3):

`tokens → embeddings → N × Transformer blocks → LayerNorm → linear head`

Sweep-tuned fields: `n_layers`, `d_model`, `n_heads`, `dropout`.


In [3]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.register_buffer("mask", None, persistent=False)

    def _causal_mask(self, size: int, device: torch.device):
        if self.mask is None or self.mask.size(0) < size:
            mask = torch.tril(torch.ones(size, size, device=device)).view(1, 1, size, size)
            self.mask = mask
        return self.mask[:, :, :size, :size]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        def reshape_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q = reshape_heads(q)
        k = reshape_heads(k)
        v = reshape_heads(v)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self._causal_mask(T, x.device)
        att = att.masked_fill(mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        return y


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff_mult: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff_mult * d_model),
            nn.GELU(),
            nn.Linear(d_ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, config: dict):
        super().__init__()
        d_model = config["d_model"]
        n_heads = config["n_heads"]
        n_layers = config["n_layers"]
        dropout = config["dropout"]
        block_size = config["block_size"]
        d_ff_mult = config.get("d_ff", 4)

        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff_mult, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.size()
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}")

        pos = torch.arange(0, T, device=idx.device, dtype=torch.long).unsqueeze(0)
        tok_emb = self.token_embedding(idx)
        pos_emb = self.pos_embedding(pos)
        x = self.drop(tok_emb + pos_emb)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss


## Learning Rate Schedule

**Linear warmup → cosine decay** down to `min_lr`. Standard schedule for transformer training; `warmup_steps` is part of the sweep.


In [4]:
class CosineWarmupScheduler(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_steps: int, max_steps: int, min_lr: float = 0.0, last_epoch: int = -1):
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = self.last_epoch + 1
        if step < self.warmup_steps:
            scale = step / max(1, self.warmup_steps)
        else:
            progress = (step - self.warmup_steps) / max(1, self.max_steps - self.warmup_steps)
            progress = min(max(progress, 0.0), 1.0)
            scale = 0.5 * (1.0 + math.cos(math.pi * progress))
        return [self.min_lr + (base_lr - self.min_lr) * scale for base_lr in self.base_lrs]


## Training Loop

| Function | Role |
|----------|------|
| `create_model_and_optimizer` | Build GPT + AdamW (decoupled weight decay) |
| `evaluate_loss` | Mean validation cross-entropy |
| `train` | W&B sweep callback — logs metrics, saves best checkpoint |

**Important:** `train()` merges `wandb.config` (sweep suggestions) over `BASE_CONFIG` *after* `wandb.init()`, so each trial actually uses the sampled hyperparameters.


In [5]:
def create_model_and_optimizer(config: dict):
    model = GPTLanguageModel(vocab_size=vocab_size, config=config).to(device)

    decay_params, no_decay_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() >= 2 and "bias" not in name and "norm" not in name.lower():
            decay_params.append(p)
        else:
            no_decay_params.append(p)

    optim_groups = [
        {"params": decay_params, "weight_decay": config["weight_decay"]},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

    optimizer = torch.optim.AdamW(
        optim_groups,
        lr=config["learning_rate"],
        betas=config.get("betas", (0.9, 0.95)),
    )

    scheduler = CosineWarmupScheduler(
        optimizer,
        warmup_steps=config["warmup_steps"],
        max_steps=config["max_steps"],
        min_lr=config.get("min_lr", 0.0),
    )

    return model, optimizer, scheduler


@torch.no_grad()
def evaluate_loss(model: nn.Module, config: dict, num_batches: int = 50):
    model.eval()
    losses = []
    for _ in range(num_batches):
        x, y = get_batch("val", batch_size=config["batch_size"], block_size=config["block_size"], device=device)
        _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def train(config: dict | None = None):
    run = wandb.init(
        project=BASE_CONFIG["wandb_project"],
        entity=BASE_CONFIG.get("wandb_entity"),
        job_type=BASE_CONFIG.get("wandb_job_type", "training-sweep"),
        config=BASE_CONFIG,
    )
    # Merge sweep-sampled hyperparameters from wandb.config over defaults.
    config = {**BASE_CONFIG, **(config or {}), **dict(wandb.config)}
    set_seed(config["seed"])

    model, optimizer, scheduler = create_model_and_optimizer(config)
    wandb.log({"model/parameters": sum(p.numel() for p in model.parameters())})

    max_steps = config["max_steps"]
    eval_interval = config["eval_interval"]
    best_val_loss = float("inf")
    best_state = None
    start_time = time.time()

    for step in range(max_steps):
        x, y = get_batch("train", batch_size=config["batch_size"], block_size=config["block_size"], device=device)

        _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        tokens_seen = (step + 1) * config["batch_size"] * config["block_size"]
        elapsed = time.time() - start_time

        log_payload = {
            "train/loss": loss.item(),
            "train/step": step,
            "train/tokens": tokens_seen,
            "train/tokens_per_sec": tokens_seen / max(elapsed, 1e-6),
            "optim/lr": optimizer.param_groups[0]["lr"],
        }

        if (step + 1) % eval_interval == 0 or step == max_steps - 1:
            val_loss = evaluate_loss(model, config)
            log_payload["val/loss"] = val_loss
            log_payload["val/perplexity"] = math.exp(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {"model": model.state_dict(), "config": config}

        wandb.log(log_payload)

    if best_state is not None:
        ckpt_dir = DATA_DIR / "checkpoints"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        ckpt_path = ckpt_dir / "gpt_best_sweep.pt"
        torch.save(best_state, ckpt_path)

        artifact = wandb.Artifact("tinyshakespeare-char-gpt", type="model")
        artifact.add_file(str(ckpt_path))
        artifact.add_file(str(vocab_path))
        run.log_artifact(artifact)

    run.finish()



## W&B Sweep Configuration

Bayesian optimization (`method: bayes`) minimizes **`val/loss`**.

| Parameter | Search | Notes |
|-----------|--------|-------|
| `learning_rate` | log-uniform [1e-4, 1e-2] | AdamW step size |
| `n_layers` | {4, 5, 6, 7, 8} | Model depth |
| `d_model` | {128, 256, 384, 512} | Hidden size |
| `n_heads` | {4, 8} | Attention heads |
| `dropout` | uniform [0.1, 0.3] | Regularization |
| `weight_decay` | {0.05, 0.1, 0.2} | AdamW decay |
| `warmup_steps` | {50, 100, 200} | LR warmup length |


In [6]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val/loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e-2},
        "n_layers": {"values": [4, 5, 6, 7, 8]},
        "d_model": {"values": [128, 256, 384, 512]},
        "n_heads": {"values": [4, 8]},
        "dropout": {"distribution": "uniform", "min": 0.1, "max": 0.3},
        "weight_decay": {"values": [0.05, 0.1, 0.2]},
        "warmup_steps": {"values": [50, 100, 200]},
        "wandb_group": {"values": ["sweep_lr_dropout", "sweep_model_size"]},
    },
}


## Create the Sweep

Registers the search space on W&B and returns a `sweep_id`. Re-run only when you change `sweep_config` — each call creates a new sweep.


In [7]:
sweep_id = wandb.sweep(sweep_config, project=BASE_CONFIG["wandb_project"], entity=BASE_CONFIG.get("wandb_entity"))
print("sweep_id:", sweep_id)

# wandb.agent(sweep_id, function=train, count=10)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\utkar\_netrc.


Create sweep with ID: x1lo44yg
Sweep URL: https://wandb.ai/utki007-northwestern-university/genre-story-generator/sweeps/x1lo44yg
sweep_id: x1lo44yg


## Run the Sweep Agent

`wandb.agent(sweep_id, function=train, count=N)` runs **N trials** locally. Each trial is a full training run (2000 steps by default).

Monitor progress: [W&B sweep dashboard](https://wandb.ai/utki007-northwestern-university/genre-story-generator/sweeps/x1lo44yg)


In [8]:
wandb.agent(sweep_id, function=train, count=10)


wandb: Agent Starting Run: fqls08t6 with config:
wandb: 	d_model: 512
wandb: 	dropout: 0.137200153398574
wandb: 	learning_rate: 0.0003018187493128268
wandb: 	n_heads: 8
wandb: 	n_layers: 6
wandb: 	wandb_group: sweep_model_size
wandb: 	warmup_steps: 50
wandb: 	weight_decay: 0.1
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\utkar\_netrc.
wandb: Currently logged in as: utki007 (utki007-northwestern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


model/parameters,▁
optim/lr,▄▆████████▇▇▇▇▇▆▆▆▆▆▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▇▅▄▄▄▃▃▃▃▃▃▃▃▃▂▃▃▂▂▂▁▂▂▂▂▂▁▁▂▁▁▁▂▁▁▁▁▁▁
train/step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇███
train/tokens,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇█
train/tokens_per_sec,▁▆▇▇▇▇▇██▇██████████████████████████████
val/loss,█▆▅▄▃▂▂▁▁▁
val/perplexity,█▆▅▄▃▂▂▁▁▁
model/parameters,826368
optim/lr,1e-05
train/loss,2.23598


wandb: Agent Starting Run: 70r7tuzc with config:
wandb: 	d_model: 512
wandb: 	dropout: 0.2447798877335061
wandb: 	learning_rate: 0.0006718207855175644
wandb: 	n_heads: 4
wandb: 	n_layers: 4
wandb: 	wandb_group: sweep_model_size
wandb: 	warmup_steps: 50
wandb: 	weight_decay: 0.05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\utkar\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


model/parameters,▁
optim/lr,▁▅████▇▇▇▇▇▇▇▆▆▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/loss,█▆▆▅▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇███
train/tokens,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/tokens_per_sec,▂▆███▄▄▄▄▄▃▂▂▃▁▂▂▂▂▃▃▃▃▃▃▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃
val/loss,█▆▅▄▃▂▂▁▁▁
val/perplexity,█▆▅▄▃▂▂▁▁▁
model/parameters,826368
optim/lr,1e-05
train/loss,2.23598


wandb: Agent Starting Run: 1nh50cfc with config:
wandb: 	d_model: 128
wandb: 	dropout: 0.10469342056597349
wandb: 	learning_rate: 0.004743993304548884
wandb: 	n_heads: 8
wandb: 	n_layers: 7
wandb: 	wandb_group: sweep_lr_dropout
wandb: 	warmup_steps: 50
wandb: 	weight_decay: 0.2
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\utkar\_netrc.


...

model/parameters,▁
optim/lr,▂███████▇▇▆▆▆▆▆▅▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train/loss,█▅▅▄▄▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/step,▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/tokens,▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█
train/tokens_per_sec,███▂▄▅▃▂▃▃▄▄▁▁▂▃▃▂▃▃▃▃▄▄▃▄▄▄▃▃▄▃▃▃▃▂▂▂▂▂
val/loss,█▆▅▄▃▂▂▁▁▁
val/perplexity,█▆▅▄▃▂▂▁▁▁
model/parameters,826368
optim/lr,1e-05
train/loss,2.23598


wandb: Agent Starting Run: xhpe9tvg with config:
wandb: 	d_model: 128
wandb: 	dropout: 0.2604904160931901
wandb: 	learning_rate: 0.00012496750836083235
wandb: 	n_heads: 8
wandb: 	n_layers: 6
wandb: 	wandb_group: sweep_lr_dropout
wandb: 	warmup_steps: 50
wandb: 	weight_decay: 0.1
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\utkar\_netrc.


model/parameters,▁
optim/lr,▃▅▅▆██████▇▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
train/loss,█▆▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁▂▁▁▂▁▁▁▁▁▂▁▂
train/step,▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇██
train/tokens,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇██
train/tokens_per_sec,▁▁▆██▃▃▄▅▃▄▅▅▆▆▃▃▃▄▃▄▄▃▄▄▄▃▃▃▃▃▄▃▃▃▃▃▂▂▂
val/loss,█▆▅▄▃▂▂▁▁▁
val/perplexity,█▆▅▄▃▂▂▁▁▁
model/parameters,826368
optim/lr,1e-05
train/loss,2.23598


wandb: Agent Starting Run: 21kujlel with config:
wandb: 	d_model: 128
wandb: 	dropout: 0.1930370007752032
wandb: 	learning_rate: 0.0010025855742916497
wandb: 	n_heads: 8
wandb: 	n_layers: 8
wandb: 	wandb_group: sweep_model_size
wandb: 	warmup_steps: 50
wandb: 	weight_decay: 0.05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\utkar\_netrc.


model/parameters,▁
optim/lr,▂▃▅▅███▇▇▇▇▇▇▇▆▆▆▅▅▅▅▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
train/loss,█▅▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/step,▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█████
train/tokens,▁▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█████
train/tokens_per_sec,██▆▆▁▃▃▃▄▁▃▄▅▅▅▆▆▄▅▅▅▆▅▅▅▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val/loss,█▆▅▄▃▂▂▁▁▁
val/perplexity,█▆▅▄▃▂▂▁▁▁
model/parameters,826368
optim/lr,1e-05
train/loss,2.23598


## Sweep Results (10 trials)

All 10 completed runs in sweep `x1lo44yg` converged to the **same** validation loss:

| Metric | Value |
|--------|-------|
| **val/loss** | **2.148** |
| **val/perplexity** | **~8.57** |
| **train/loss** (final) | ~2.24 |

### Why every run looks identical

The first sweep had a bug: `train()` built the model from `BASE_CONFIG` *before* reading `wandb.config`, so **every trial trained the same small model** (d_model=128, n_layers=4, ~826K params) regardless of what the sweep sampled. W&B logged the suggested hyperparameters, but they were never applied.

The cell above fixes this by merging `dict(wandb.config)` after `wandb.init()`.

### Effective config used in the completed sweep

Because of the bug, the results reflect **`BASE_CONFIG`**, not the sampled values:

| Hyperparameter | Value |
|----------------|-------|
| `learning_rate` | 3e-4 |
| `n_layers` | 4 |
| `d_model` | 128 |
| `n_heads` | 4 |
| `dropout` | 0.2 |
| `weight_decay` | 0.1 |
| `warmup_steps` | 100 |

### Recommendation

1. **Re-run the sweep** with the fixed `train()` so trials actually explore the search space.
2. Until then, use the baseline config above — it achieves **val/loss ≈ 2.15** (perplexity ≈ 8.6) in 2000 steps.
3. After a proper re-sweep, pick the run with lowest `val/loss` on the [leaderboard](https://wandb.ai/utki007-northwestern-university/genre-story-generator/sweeps/x1lo44yg); prefer smaller models when validation loss ties.


## Notes

- Requires `data/artifacts/` from notebook 2.
- Best checkpoint per trial: `data/checkpoints/gpt_best_sweep.pt`
- W&B artifact: `tinyshakespeare-char-gpt`
- Compare runs by `wandb_group` in the W&B UI
